# P1 — Task 1 — EDA + slide Giới thiệu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task1_eda.ipynb)

**Slide sở hữu:** 4, 5, 6, 7 · **Slide mới:** +1: Q3 kênh theo tỉnh  
**Đã có:** task1_eda.py, task1_findings.md, chart t1_monthly_spend / t1_essential_discretionary / t1_category_ticket / t1_age_cohort

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Code đầy đủ: `draft/task1_deep.py` (chạy độc lập được). Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, re, sys, gzip, shutil, urllib.request
REPO = "/content/BI10GroupYAPPERS"
# Data tải thẳng từ thư mục Drive public (anyone with link) — không cần mount Drive / Add shortcut
# Thư mục: https://drive.google.com/drive/folders/1gOqCyY9IfFUJtbr2r3vR_Z3t61O71CrC
URL = {
    "consumer_financial_health_engagement_2025.csv": "https://drive.google.com/file/d/1YyueAH9Kasid6V_obml9pCMwh2dB0ryn/view",
    "consumer_transactions_2025.parquet.gz":         "https://drive.google.com/file/d/1O33KpFbUx6_BRGwFTb80xVSB3tzXqt5H/view",
    "data_dictionary.xlsx":                          "https://drive.google.com/file/d/1nFlZHmuaoK8VQ5xqRN2f4ZLqoR3gk2j1/view",
}

from google.colab import userdata

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f, u in URL.items():
    if not os.path.exists(f"{D}/{f}"):
        fid = re.search(r"/d/([\w-]+)", u).group(1)   # tải file public bằng stdlib, không cần gdown
        urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", f"{D}/{f}")
if not os.path.exists(TX):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
    with gzip.open(f"{D}/consumer_transactions_2025.parquet.gz") as src, open(TX, "wb") as dst:
        shutil.copyfileobj(src, dst)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task1_output.txt`.

In [ ]:
%run -i draft/task1_eda.py

In [ ]:
# Style chung cho chart (giống draft/make_charts.py)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 250, "display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 11, "axes.titlesize": 15, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25})
C = dict(primary="#2E5EAA", accent="#E4572E", good="#3C9D6E", warn="#E0A32E", muted="#9AA5B1", ink="#22303F")
os.makedirs("outputs/figures", exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"outputs/figures/{name}", dpi=150, bbox_inches="tight"); plt.show(); print("saved", name)
def h(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)
# cột bổ sung cho Q1/Q4 (task1_eda.py chỉ nạp 6 cột)
tx2 = pd.read_csv(TXN, usecols=["consumer_id", "activity_datetime", "spending_category", "transaction_month"])


## Việc 1 — Q3 — bảng kênh chi tiết theo tỉnh (SLIDE MỚI)  `[BẮT BUỘC · hạn 27/09 12:00]`
Đề bắt 'compare channel breakdown (POS vs digital)' cho ≥2 tỉnh. Với Hà Nội, Đồng Nai, Lâm Đồng, Hưng Yên, Hải Phòng + toàn quốc: tỷ trọng SỐ GIAO DỊCH và GIÁ TRỊ theo POS / QR / E-com / Mobile / Recurring. Chart stacked bar 100% ngang (6 thanh). Kết luận giữ: chênh ±0.5pp → địa lý không phải đòn bẩy.

**Giao:** Chart t1_province_channel.png + bullet slide mới trong slide_outline.md

In [ ]:
h("DEEP Q3 - Channel breakdown for high-spend, below-average-digital provinces")
CH = ["POS", "QR Payment", "E-commerce", "Mobile App", "Recurring Payment"]
top = target.head(5)["province_city"].tolist()
sub = txn[txn.province_city.isin(top)]
cnt = pd.crosstab(sub.province_city, sub.transaction_channel, normalize="index")[CH] * 100
val = pd.crosstab(sub.province_city, sub.transaction_channel, values=sub.spend_amount_vnd, aggfunc="sum", normalize="index")[CH] * 100
nat_c = txn.transaction_channel.value_counts(normalize=True)[CH] * 100
nat_v = txn.groupby("transaction_channel").spend_amount_vnd.sum()[CH] / txn.spend_amount_vnd.sum() * 100
cnt.loc["National"], val.loc["National"] = nat_c, nat_v
cnt, val = cnt.loc[top + ["National"]], val.loc[top + ["National"]]
print("Share of TRANSACTIONS by channel (%):\n", cnt.round(1).to_string())
print("\nShare of SPEND VALUE by channel (%):\n", val.round(1).to_string())
gap = (cnt.drop("National")[CH[1:]].sum(1) - cnt.loc["National", CH[1:]].sum()).round(2)
print("\nDigital-share gap vs national (pp, by count):\n", gap.to_string())
print(f"-> max |gap| = {gap.abs().max():.2f}pp: channel mix is near-identical; digital value share "
      f"{val.drop('National')[CH[1:]].sum(1).min():.1f}-{val.drop('National')[CH[1:]].sum(1).max():.1f}% vs national {val.loc['National', CH[1:]].sum():.1f}%")

EN = {"Thành phố Hồ Chí Minh": "Ho Chi Minh City", "Hà Nội": "Ha Noi", "Đồng Nai": "Dong Nai", "Lâm Đồng": "Lam Dong",
      "Hưng Yên": "Hung Yen", "Hải Phòng": "Hai Phong", "Cần Thơ": "Can Tho", "Đà Nẵng": "Da Nang", "National": "National avg"}
fig, ax = plt.subplots(figsize=(10, 5.5))
cols = [C["muted"], C["primary"], C["good"], C["warn"], C["accent"]]
left = np.zeros(len(cnt))
for ch, col in zip(CH, cols):
    ax.barh([EN.get(p, p) for p in cnt.index], cnt[ch], left=left, color=col, label=ch)
    for i, v in enumerate(cnt[ch]):
        if v > 4: ax.text(left[i] + v / 2, i, f"{v:.1f}", ha="center", va="center", color="white", fontsize=9)
    left += cnt[ch].values
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("% of transactions"); ax.grid(axis="y", visible=False)
ax.set_title("High-spend provinces: channel mix mirrors the national average")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(.5, -.12), frameon=False, fontsize=9)
save(fig, "t1_province_channel.png")


## Việc 2 — Q2 — chứng minh độ dốc, không chỉ 2 đầu  `[Cao · hạn 27/09 12:00]`
Chia FHS thành bin 10 điểm, tính tỷ trọng discretionary từng bin → line chart. Đường giảm đều = 'đảo chiều liên tục, không do ngưỡng 40/80'. Thử thêm ngưỡng FHS<45 vs ≥75 để chứng minh độ bền.

**Giao:** Line chart + 1 bullet slide 6

In [ ]:
h("DEEP Q2 - Is the essential->discretionary inversion a gradient or a threshold artefact?")
mon["fhs_bin"] = pd.cut(mon.financial_health_score, [0, 40, 50, 60, 70, 80, 90, 101], right=False,
                        labels=["<40", "40-49", "50-59", "60-69", "70-79", "80-89", "90+"])
gb = mon.groupby("fhs_bin", observed=True).apply(lambda d: pd.Series({
    "months": len(d), "discretionary_pct": 100 * d.discretionary_spend_vnd.sum() / (d.essential_spend_vnd.sum() + d.discretionary_spend_vnd.sum())}))
print(gb.round(1).to_string())
print(f"Spearman(FHS, discretionary_spend_ratio) month grain: "
      f"{mon.financial_health_score.corr(mon.discretionary_spend_ratio, method='spearman'):+.3f}")
for lo, hi in [(35, 85), (40, 80), (45, 75), (50, 70)]:
    a, b = mon[mon.financial_health_score < lo], mon[mon.financial_health_score >= hi]
    da = 100 * a.discretionary_spend_vnd.sum() / (a.essential_spend_vnd.sum() + a.discretionary_spend_vnd.sum())
    db = 100 * b.discretionary_spend_vnd.sum() / (b.essential_spend_vnd.sum() + b.discretionary_spend_vnd.sum())
    print(f"  FHS<{lo} ({len(a):>5} mo) {da:5.1f}% discretionary  vs  FHS>={hi} ({len(b):>5} mo) {db:5.1f}%  -> gap {da-db:+.1f}pp")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(gb.index.astype(str), gb.discretionary_pct, marker="o", lw=2.5, color=C["accent"])
for x, y, n in zip(gb.index.astype(str), gb.discretionary_pct, gb.months):
    ax.annotate(f"{y:.0f}%\n(n={int(n):,})", (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=9)
ax.axhline(50, color=C["muted"], ls="--", lw=1); ax.set_ylim(0, 100)
ax.set_xlabel("Financial health score band (consumer-month)"); ax.set_ylabel("Discretionary share of spend (%)")
ax.set_title("Discretionary share falls steadily as health rises")
save(fig, "t1_discretionary_gradient.png")


## Việc 3 — Q1 — tháng 12 tăng vì category nào?  `[Trung bình · hạn 27/09 18:00]`
Mức tăng số giao dịch T12 vs T2 theo category → top 5 đóng góp. Kiểm tra caveat 'gộp 2 năm vào 2025' bằng timestamp gốc (có năm 2024 không?) và ghi rõ kết quả.

**Giao:** 2 dòng findings + bullet slide 5

In [ ]:
h("DEEP Q1 - Which categories drive the December peak? + 2-year fold check")
g = tx2.groupby(["transaction_month", "spending_category"]).size().unstack(fill_value=0)
inc = (g.loc[12] - g.loc[2]).sort_values(ascending=False)
print(f"Dec - Feb transaction count, total increase {inc.sum():,}:")
print(pd.DataFrame({"increase": inc, "share_of_increase_%": (100 * inc / inc.sum()).round(1),
                    "growth_%": (100 * (g.loc[12] / g.loc[2] - 1)).round(0)}).head(14).to_string())
yr = pd.to_datetime(tx2.activity_datetime).dt.year.value_counts().sort_index()
print("\nactivity_datetime years:\n", yr.to_string())
dec = pd.to_datetime(tx2.loc[tx2.transaction_month == 12, "activity_datetime"]).dt.year.value_counts().sort_index()
print("December rows by year:\n", dec.to_string())


## Việc 4 — Q4 — bổ sung chỉ số thói quen  `[Thấp · hạn 27/09 18:00]`
% khách có mua Fuel / Groceries; số lần mua trung bình / khách / tháng cho 2 category top.

**Giao:** Bullet slide 7

In [ ]:
h("DEEP Q4 - Reach & habit of the two top categories")
top_cnt_cat, top_spend_cat = cat["count"].idxmax(), cat["sum"].idxmax()
nmon = tx2.groupby("consumer_id").transaction_month.nunique()
for c_ in [top_cnt_cat, top_spend_cat]:
    s = tx2[tx2.spending_category == c_]
    buyers = s.consumer_id.nunique()
    per = s.groupby(["consumer_id", "transaction_month"]).size()
    print(f"{c_}: {buyers}/999 consumers buy it ({100*buyers/999:.1f}%) | "
          f"{per.mean():.1f} purchases per buyer-month | active in {s.groupby('consumer_id').transaction_month.nunique().mean():.1f} months/yr")


## Việc 5 — Q5 — tuổi là tín hiệu yếu (bootstrap CI 95%)  `[Trung bình · hạn 27/09 18:00]`
Bootstrap 1000 lần FHS trung bình từng nhóm tuổi (giữ 7 khách 15–17 tuổi). Nếu các CI chồng nhau → nói rõ 'tuổi không phân biệt được'. Gửi bảng CI cho P2 dùng chung.

**Giao:** Bảng CI (gửi P2) + bullet slide 7

In [ ]:
h("DEEP Q5 - Bootstrap 95% CI of mean FHS per age cohort (consumer level, all 999 incl. 7 minors)")
cons1 = mon.groupby("consumer_id").agg(fhs=("financial_health_score", "mean"), age=("age", "first"))
cons1["cohort"] = pd.cut(cons1.age, bins=bins, labels=labels)
rng = np.random.default_rng(42)
rows = []
for k, s in cons1.groupby("cohort", observed=True).fhs:
    bs = [rng.choice(s.values, len(s)).mean() for _ in range(1000)]
    rows.append((k, len(s), s.mean(), *np.percentile(bs, [2.5, 97.5])))
ci = pd.DataFrame(rows, columns=["cohort", "n", "mean_fhs", "ci_lo", "ci_hi"]).set_index("cohort")
print(ci.round(2).to_string())
lo_max, hi_min = ci.ci_lo.max(), ci.ci_hi.min()
print(f"All CIs overlap: {bool(lo_max <= hi_min)} (max lower {lo_max:.2f} vs min upper {hi_min:.2f})")
ci.to_csv("draft/task1_age_ci.csv")   # P2 dùng chung
assert ci.n.sum() == 999


## Việc 6 — Slide 4 — rà số dataset  `[Thấp · hạn 27/09 18:00]`
Kiểm lại 999 khách / 1,852,394 giao dịch / 10,992 consumer-month và 2 luật (tỷ lệ, wellbeing).

**Giao:** Xác nhận cho P5

In [ ]:
h("Slide 4 dataset check")
print(f"consumer-months {len(mon):,} | consumers {mon.consumer_id.nunique()} | transactions {len(txn):,} | "
      f"min age {mon.age.min()} | minors {mon.loc[mon.age < 18].consumer_id.nunique()}")
assert (len(mon), mon.consumer_id.nunique(), len(txn)) == (10992, 999, 1852394)


## Findings — kết quả đã chạy 27/09 (chi tiết: `draft/task1_deep_output.txt`)
- Q3: 5 tỉnh chi nhiều nhưng digital thấp hơn TB (Hà Nội, Đồng Nai, Lâm Đồng, Hưng Yên, Hải Phòng) lệch tối đa **0.46pp** so với toàn quốc; POS 58.9–59.3% vs 58.8% → địa lý không phải đòn bẩy.
- Q2: tỷ trọng discretionary giảm đều theo FHS: **71.6% (<40) → 64.2 → 57.3 → 54.4 → 49.2 → 40.6% (80–89)**; ngưỡng nào cũng giữ chiều (FHS<35 vs ≥85: 73.5% vs 30.2%) → đảo chiều là một độ dốc, không do ngưỡng.
- Q1: T12 tăng **đồng đều ở cả 14 category (+181–193%)**, không category nào dẫn dắt; 100% timestamp là năm 2025 → bỏ caveat 'gộp 2 năm', thay bằng 'mức tăng đồng đều giống hệ số mùa vụ của dữ liệu tổng hợp'.
- Q4: Fuel được **96.8%** khách mua, 17.3 lần/khách-tháng; Groceries **99.3%**, 16.1 lần → cả hai là thói quen gần như toàn dân.
- Q5: CI 95% FHS của 6 nhóm tuổi **chồng nhau** (65.75–66.97) → tuổi không phân biệt được sức khoẻ tài chính. Bảng CI lưu ở draft/task1_age_ci.csv cho P2.

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task1_eda.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.